In [1]:
# Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
data = np.load('data/processed_data.npz')
X_train = data['X_train']
X_val = data['X_val']
X_test = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']
snr_test = data['snr_test']

print(f"Train: {X_train.shape}")

Using device: cpu
Train: (158400, 2, 128)


In [2]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=2, hidden_size=128, num_layers=2, num_classes=11):
        super(LSTMModel, self).__init__()
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=0.5)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x shape: (batch, 2, 128) -> need (batch, 128, 2) for LSTM
        x = x.permute(0, 2, 1)  # (batch, seq_len=128, features=2)
        
        # LSTM output
        out, (h_n, c_n) = self.lstm(x)
        
        # Use last hidden state
        out = self.fc(h_n[-1])
        return out

# Test
model = LSTMModel().to(device)
dummy = torch.randn(4, 2, 128).to(device)
out = model(dummy)
print(f"Model output shape: {out.shape}")
print(f"\nModel architecture:")
print(model)

Model output shape: torch.Size([4, 11])

Model architecture:
LSTMModel(
  (lstm): LSTM(2, 128, num_layers=2, batch_first=True, dropout=0.5)
  (fc): Linear(in_features=128, out_features=11, bias=True)
)


In [ ]:
# Faster LSTM - smaller hidden size, fewer epochs
class LSTMModel(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=1, num_classes=11):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, (h_n, c_n) = self.lstm(x)
        out = self.dropout(h_n[-1])
        out = self.fc(out)
        return out

model = LSTMModel().to(device)
history = train_model(model, train_loader, val_loader, epochs=10, lr=0.001)

torch.save(model.state_dict(), 'models/lstm_model.pt')
print("\nModel saved.")

Epoch 1/10:   2%|█▎                                                                  | 25/1238 [00:04<03:09,  6.42it/s]

In [1]:
# Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cpu')

# Load data
data = np.load('data/processed_data.npz')
X_train = torch.FloatTensor(data['X_train'])
y_train = torch.LongTensor(data['y_train'])
X_val = torch.FloatTensor(data['X_val'])
y_val = torch.LongTensor(data['y_val'])

# Use smaller subset for faster training
X_train_small = X_train[:50000]
y_train_small = y_train[:50000]

train_loader = DataLoader(TensorDataset(X_train_small, y_train_small), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)

print(f"Training on {len(X_train_small)} samples")

Training on 50000 samples


In [2]:
# Simple LSTM
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(2, 64, num_layers=1, batch_first=True)
        self.fc = nn.Linear(64, 11)
        
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (batch, 128, 2)
        out, (h_n, c_n) = self.lstm(x)
        out = self.fc(h_n[-1])
        return out

model = LSTMModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train for 10 epochs
for epoch in range(10):
    model.train()
    total_loss, correct = 0, 0
    
    for X_batch, y_batch in tqdm(train_loader, desc=f'Epoch {epoch+1}', leave=True):
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == y_batch).sum().item()
    
    train_acc = correct / len(train_loader.dataset)
    
    # Quick val check
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            out = model(X_batch)
            val_correct += (out.argmax(1) == y_batch).sum().item()
    val_acc = val_correct / len(val_loader.dataset)
    
    print(f'  Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

torch.save(model.state_dict(), 'models/lstm_model.pt')
print("\nDone! Model saved.")

Epoch 1: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:37<00:00,  5.25it/s]


  Train Acc: 0.0921, Val Acc: 0.0909


Epoch 2: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:37<00:00,  5.26it/s]


  Train Acc: 0.0920, Val Acc: 0.0907


Epoch 3: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:36<00:00,  5.42it/s]


...

  Train Acc: 0.0947, Val Acc: 0.0909


Epoch 9: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:21<00:00,  9.19it/s]


  Train Acc: 0.0891, Val Acc: 0.0903


Epoch 10: 100%|██████████████████████████████████████████████████████████████████████| 196/196 [00:22<00:00,  8.79it/s]


  Train Acc: 0.1004, Val Acc: 0.1114

Done! Model saved.


In [3]:
# Hybrid CNN-LSTM
class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        # CNN for feature extraction
        self.conv1 = nn.Conv1d(2, 64, kernel_size=8, padding='same')
        self.pool = nn.MaxPool1d(2)
        
        # LSTM on CNN features
        self.lstm = nn.LSTM(64, 64, num_layers=1, batch_first=True)
        self.fc = nn.Linear(64, 11)
        
    def forward(self, x):
        # CNN: (batch, 2, 128) -> (batch, 64, 64)
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        
        # Reshape for LSTM: (batch, 64, 64) -> (batch, seq=64, features=64)
        x = x.permute(0, 2, 1)
        
        # LSTM
        out, (h_n, c_n) = self.lstm(x)
        out = self.fc(h_n[-1])
        return out

model = HybridModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    correct = 0
    for X_batch, y_batch in tqdm(train_loader, desc=f'Epoch {epoch+1}', leave=True):
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == y_batch).sum().item()
    
    train_acc = correct / len(train_loader.dataset)
    
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            val_correct += (model(X_batch).argmax(1) == y_batch).sum().item()
    val_acc = val_correct / len(val_loader.dataset)
    
    print(f'  Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

torch.save(model.state_dict(), 'models/hybrid_model.pt')
print("\nDone! Model saved.")

Epoch 1:   0%|                                                                                 | 0/196 [00:00<?, ?it/s]C:\Users\nabee\anaconda3\envs\amc\lib\site-packages\torch\nn\modules\conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1025.)
  return F.conv1d(
Epoch 1: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:23<00:00,  8.29it/s]


  Train Acc: 0.0975, Val Acc: 0.0932


Epoch 2: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:23<00:00,  8.20it/s]


  Train Acc: 0.1085, Val Acc: 0.1238


Epoch 3: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:26<00:00,  7.43it/s]


...

  Train Acc: 0.1827, Val Acc: 0.1765


Epoch 9: 100%|███████████████████████████████████████████████████████████████████████| 196/196 [00:28<00:00,  6.99it/s]


  Train Acc: 0.1970, Val Acc: 0.2035


Epoch 10: 100%|██████████████████████████████████████████████████████████████████████| 196/196 [00:27<00:00,  7.08it/s]


  Train Acc: 0.2020, Val Acc: 0.2073

Done! Model saved.
